# Clase 5 · De los datos al modelo: un proyecto completo

**Machine Learning** · Programa de Especialización Data Scientist · Cierre del Módulo 1 · Sesión 5 de 16

Docente: **Josef Renato Rodriguez**

---

## Por qué esta clase

En las Clases 2, 3 y 4 vimos siete modelos de regresión, siempre con el mismo dataset y con la
preparación ya hecha. En un proyecto real, el modelo es la parte corta: antes hay que entender el
problema, revisar los datos, decidir qué se limpia, crear variables y armar todo para que no haya
trampas. Después hay que medir bien y dejarlo listo para usar.

Hoy hacemos **un proyecto completo, de punta a punta**, con datos que no conoces: ventas reales de
viviendas en Ames, Iowa (EE. UU.). Es el mismo problema de **tasación de garantías**, pero esta vez
con casas individuales y 22 variables por casa.

## Los 8 pasos de hoy (y de tu tarea)

| Paso | Qué haces | Min |
|---|---|---|
| 1 | Definir el problema y la métrica | 3 |
| 2 | Cargar y revisar los datos | 6 |
| 3 | Separar la prueba **antes** de mirar | 4 |
| 4 | Explorar, solo con entrenamiento | 7 |
| 5 | Limpiar y crear variables | 8 |
| 6 | Armar el pipeline | 7 |
| 7 | Comparar modelos con validación cruzada | 12 |
| 8 | Evaluar una vez en prueba y dejarlo listo para usar | 11 |

Esta lista es la **plantilla de la tarea**: vas a repetir los 8 pasos con otro dataset.

Igual que siempre: completa las celdas **`TU CÓDIGO`** y ejecuta la **`VERIFICACIÓN`** de abajo.

---
## Celda 0 · Librerías y funciones de apoyo

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             mean_absolute_percentage_error)
import warnings
warnings.filterwarnings("ignore")

SEED = 42
NAVY, BLUE, MAG, GREEN, GRAY = "#0A2559", "#1A56E8", "#E6115E", "#12B886", "#5B6B87"
plt.rcParams.update({
    "figure.figsize": (7, 4.2), "figure.dpi": 110,
    "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 11,
})

def check(nombre, obtenido, esperado, tol=1e-3):
    if obtenido is None or (isinstance(obtenido, float) and np.isnan(obtenido)):
        print(f"[ ] {nombre}: todavía no calculaste nada")
        return False
    ok = abs(float(obtenido) - float(esperado)) <= tol
    print(f"{'[OK]' if ok else '[X ]'} {nombre}")
    print(f"     tu resultado: {float(obtenido):.4f}   |   esperado: {float(esperado):.4f}")
    return ok

def check_bool(nombre, cond, pista=""):
    print(f"{'[OK]' if cond else '[X ]'} {nombre}")
    if not cond and pista:
        print(f"     pista: {pista}")
    return bool(cond)

def metricas(y_real, y_pred):
    return {"MAE": mean_absolute_error(y_real, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_real, y_pred)),
            "R2": r2_score(y_real, y_pred),
            "MAPE": mean_absolute_percentage_error(y_real, y_pred)}
print("Listo")

---
# Paso 1 · Definir el problema y la métrica  ·  3 min

Antes de abrir los datos, cuatro preguntas. Si no puedes responderlas, todavía no tienes un proyecto.

| Pregunta | Respuesta para hoy |
|---|---|
| ¿Qué decisión va a apoyar el modelo? | Tasar una vivienda: estimar su precio de venta antes de aceptarla como garantía |
| ¿Qué se predice, y para qué unidad? | `sale_price`, el precio de **una venta** (en USD) |
| ¿Con qué se mide? | **MAE** en dólares, y **MAPE** en %: los tasadores hablan de «me equivoqué un 8 %» |
| ¿Contra qué se compara? | Un modelo trivial que siempre predice el precio mediano |

Y una quinta que se olvida casi siempre: **¿para qué casos NO aplica?** Lo vamos a definir en el Paso 4,
mirando los datos.

---
# Paso 2 · Cargar y revisar los datos  ·  6 min

Los datos son las 2 930 ventas de viviendas registradas por la oficina de tasación de Ames
entre 2006 y 2010, recopiladas por Dean De Cock (2011) como alternativa moderna al clásico dataset de
Boston. El archivo original tiene 82 columnas; nos quedamos con 22 variables, el identificador y el
precio. El diccionario completo está en `diccionario_ames_housing.md`.

La celda intenta descargar los datos de internet. Si no puede, sube `ames_housing.csv` con el botón de
archivos de Colab y vuelve a ejecutarla.

In [ ]:
URL = "https://raw.githubusercontent.com/wblakecannon/ames/master/data/housing.csv"
COLUMNAS = {"PID": "pid", "SalePrice": "sale_price", "Gr Liv Area": "gr_liv_area",
            "Total Bsmt SF": "total_bsmt_sf", "Lot Area": "lot_area", "Lot Frontage": "lot_frontage",
            "Year Built": "year_built", "Year Remod/Add": "year_remod", "Overall Qual": "overall_qual",
            "Overall Cond": "overall_cond", "Garage Cars": "garage_cars", "Garage Type": "garage_type",
            "Full Bath": "full_bath", "Half Bath": "half_bath", "Bedroom AbvGr": "bedrooms",
            "Fireplaces": "fireplaces", "Mas Vnr Area": "mas_vnr_area", "Neighborhood": "neighborhood",
            "Kitchen Qual": "kitchen_qual", "Central Air": "central_air", "Bsmt Qual": "bsmt_qual",
            "Sale Condition": "sale_condition", "MS Zoning": "ms_zoning", "Yr Sold": "yr_sold"}
try:
    casas = pd.read_csv(URL)[list(COLUMNAS)].rename(columns=COLUMNAS)
    print("Datos descargados de internet.")
except Exception as e:
    print(f"No se pudo descargar ({type(e).__name__}). Usando el archivo local.")
    casas = pd.read_csv("ames_housing.csv")

casas.head()

### Ejercicio 2.1 — La primera revisión

Las cuatro cosas que se miran siempre antes que nada: tamaño, duplicados, faltantes y la variable objetivo.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
n_filas, n_columnas = None, None        # casas.shape
pid_unico = None                        # ¿cada venta aparece una sola vez? casas["pid"].is_unique
faltantes = None                        # casas.isna().sum(), quedándote solo con las columnas > 0
pct_frontage = None                     # proporción de faltantes en lot_frontage. Pista: .isna().mean()
precio_mediano = None                   # mediana de sale_price

In [ ]:
# ── VERIFICACIÓN 2.1 ─────────────────────────────────────────────────────
r = [check("filas", n_filas, 2930, tol=0),
     check("columnas", n_columnas, 24, tol=0),
     check_bool("no hay ventas duplicadas", pid_unico is True),
     check("proporción de faltantes en lot_frontage", pct_frontage, 0.1672),
     check("precio mediano", precio_mediano, 160000, tol=0)]
print()
print("2.1 OK" if all(r) else "Revisa 2.1")

### No todos los faltantes son iguales

El diccionario de De Cock lo aclara, y cambia por completo qué hacer con cada uno:

| Columna | Faltantes | Qué significa | Qué hacemos |
|---|---|---|---|
| `garage_type` | 157 | **la casa no tiene garaje** | categoría nueva: «Sin garaje» |
| `bsmt_qual` | 80 | **la casa no tiene sótano** | calidad 0 |
| `lot_frontage` | 490 | **no se registró** (frente del lote) | rellenar con la mediana |
| `mas_vnr_area` | 23 | no se registró | rellenar con la mediana |
| `total_bsmt_sf`, `garage_cars` | 1 cada una | no se registró | rellenar con la mediana |

> Un `NaN` que significa «no tiene» es **información**, no un hueco. Rellenarlo con la mediana sería
> inventarle un garaje a 157 casas.

### Demostración 2.2 — La forma del precio

In [ ]:
# ── DEMOSTRACIÓN 2.2 ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].hist(casas["sale_price"], bins=50, color=BLUE)
axes[0].set_title(f"Precio · asimetría {casas['sale_price'].skew():.2f}")
axes[0].set_xlabel("USD")
axes[1].hist(np.log(casas["sale_price"]), bins=50, color=MAG)
axes[1].set_title(f"log(precio) · asimetría {np.log(casas['sale_price']).skew():.2f}")
axes[1].set_xlabel("log(USD)")
plt.tight_layout(); plt.show()

El precio tiene una cola larga a la derecha: muchas casas de 130 a 210 mil y pocas de más de 400 mil. En
logaritmo queda casi simétrico. Además, en tasación los errores son **proporcionales**: equivocarse en
20 mil es grave en una casa de 100 mil y razonable en una de 500 mil. En el Paso 6 vamos a entrenar los
modelos sobre `log(precio)` por las dos razones.

---
# Paso 3 · Separar la prueba antes de mirar  ·  4 min

Todo lo que decidas mirando los datos (qué limpiar, qué variable crear, qué modelo usar) es una forma
de aprender de ellos. Si lo decides mirando también la prueba, la prueba deja de ser «datos nunca
vistos». Por eso se separa **ahora**, antes de explorar.

Un detalle: con un precio tan asimétrico, un corte al azar puede dejar más casas caras de un lado. Para
evitarlo **estratificamos por quintiles de precio**: `pd.qcut(y, 5, labels=False)` pone a cada casa en uno
de 5 grupos de precio, y el corte respeta esa proporción.

### Ejercicio 3.1 — El corte 80/20

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
X = casas.drop(columns=["sale_price", "pid"])
y = casas["sale_price"]

X_train, X_test, y_train, y_test = None, None, None, None
# train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=pd.qcut(y, 5, labels=False))

In [ ]:
# ── VERIFICACIÓN 3.1 ─────────────────────────────────────────────────────
r = [check("casas de entrenamiento", None if X_train is None else len(X_train), 2344, tol=0),
     check("casas de prueba", None if X_test is None else len(X_test), 586, tol=0),
     check("mediana del precio en prueba", None if y_test is None else y_test.median(), 161875, tol=1)]
print()
print("3.1 OK" if all(r) else "Revisa 3.1")

---
# Paso 4 · Explorar, solo con entrenamiento  ·  7 min

Tres preguntas: ¿qué variables se relacionan con el precio?, ¿hay casos raros?, ¿hay categorías que
importen?

### Ejercicio 4.1 — Correlaciones y casos raros

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
correlaciones = None    # X_train.select_dtypes("number").corrwith(y_train), ordenadas de mayor a menor
var_top = None          # el nombre de la variable con mayor correlación: correlaciones.index[0]
n_grandes = None        # cuántas casas de entrenamiento tienen gr_liv_area > 4000 (pies cuadrados)

In [ ]:
# ── VERIFICACIÓN 4.1 ─────────────────────────────────────────────────────
r = [check_bool("la variable más correlacionada es overall_qual", var_top == "overall_qual",
                "ordena con sort_values(ascending=False)"),
     check("correlación de overall_qual", None if correlaciones is None else correlaciones.iloc[0], 0.7969),
     check("casas de más de 4 000 pies²", n_grandes, 4, tol=0)]
print()
print("4.1 OK" if all(r) else "Revisa 4.1")

`overall_qual` (calidad general del material y la terminación, de 1 a 10, asignada por el tasador) le gana
a la superficie. Primera lección de negocio: **el juicio experto ya está dentro de los datos**.

### Demostración 4.2 — Los casos raros

In [ ]:
# ── DEMOSTRACIÓN 4.2 ─────────────────────────────────────────────────────
if X_train is not None:
    grande = X_train["gr_liv_area"] > 4000
    fig, ax = plt.subplots(figsize=(8, 4.2))
    ax.scatter(X_train.loc[~grande, "gr_liv_area"], y_train[~grande], s=6, alpha=0.4, color=BLUE)
    ax.scatter(X_train.loc[grande, "gr_liv_area"], y_train[grande], s=60, color=MAG, label="más de 4 000 pies²")
    for k, (xx, yy, cond) in enumerate(zip(X_train.loc[grande, "gr_liv_area"], y_train[grande], X_train.loc[grande, "sale_condition"])):
        ax.annotate(cond, (xx, yy), textcoords="offset points", xytext=(6, 6 if k % 2 else -14), fontsize=9, color=MAG)
    ax.set_xlabel("superficie habitable sobre el suelo (pies²)"); ax.set_ylabel("precio (USD)")
    ax.set_title("Dos mansiones caras y dos ventas parciales muy baratas")
    ax.legend(); plt.tight_layout(); plt.show()

    print(X_train.loc[grande, ["gr_liv_area", "neighborhood", "sale_condition"]].assign(precio=y_train[grande]))

Las dos casas enormes y baratas son ventas **`Partial`**: casas nuevas vendidas antes de terminarse. No
son errores de captura, pero tampoco son como las demás. El propio De Cock recomienda **sacar del
análisis las casas de más de 4 000 pies²** (unos 370 m²).

Aquí no lo hacemos para «limpiar» el resultado, sino como una **decisión de alcance**: el modelo tasa
viviendas de hasta 4 000 pies². Las más grandes van a un tasador. La regla se aplica igual en
entrenamiento, en prueba y con cada casa nueva.

### Demostración 4.3 — Las categorías importan

In [ ]:
# ── DEMOSTRACIÓN 4.3 ─────────────────────────────────────────────────────
if X_train is not None:
    por_barrio = y_train.groupby(X_train["neighborhood"]).median().sort_values()
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(por_barrio.index, por_barrio.values, color=[MAG if v > 250000 else BLUE for v in por_barrio.values])
    ax.set_ylabel("precio mediano (USD)"); ax.set_title(f"{len(por_barrio)} barrios: del más barato al más caro")
    plt.xticks(rotation=70, fontsize=8); plt.tight_layout(); plt.show()

    print(y_train.groupby(X_train["sale_condition"]).median().sort_values().round(0).to_string())

El barrio más caro (StoneBr) tiene una mediana 3.6 veces la del más barato (MeadowV). Y las
ventas `Partial` (casas nuevas) tienen una mediana de 263 784, frente a 159 500 de las normales. Las
dos categorías entran al modelo.

---
# Paso 5 · Limpiar y crear variables  ·  8 min

Dos reglas de oro:

1. **Todo lo que transforma los datos va en una función**, para aplicarla igual a entrenamiento, a
   prueba y a cada casa nueva que llegue mañana.
2. **Nada de lo que hace esa función se aprende de los datos.** Restar dos años o sumar dos superficies
   no usa promedios. Lo que sí se aprende (medianas para rellenar, escalas) va en el Paso 6.

Primero, el alcance:

In [ ]:
# ── El alcance del modelo: viviendas de hasta 4 000 pies² ─────────────────
LIMITE = 4000
def fuera_de_alcance(df):
    return df["gr_liv_area"] > LIMITE

if X_train is not None:
    en_train, en_test = ~fuera_de_alcance(X_train), ~fuera_de_alcance(X_test)
    X_train, y_train = X_train[en_train], y_train[en_train]
    X_test, y_test = X_test[en_test], y_test[en_test]
    print(f"entrenamiento: {len(X_train)} casas · prueba: {len(X_test)} casas")

### Ejercicio 5.1 — La función `preparar`

Completa las cuatro líneas que faltan. Las demás ya están hechas.

| Variable nueva | Cómo se calcula | Por qué |
|---|---|---|
| `edad` | `yr_sold - year_built` | importa la edad al momento de la venta, no el año |
| `area_total` | `gr_liv_area + total_bsmt_sf` (con el sótano faltante como 0) | superficie habitable + sótano |
| `banos` | `full_bath + 0.5 * half_bath` | un medio baño vale medio |
| `garage_type` | rellenar `NaN` con `"Sin garaje"` | el faltante significa «no tiene» |

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
CALIDAD = {"Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}     # pobre, regular, típica, buena, excelente

def preparar(df):
    df = df.copy()
    df["edad"] = None                 # yr_sold menos year_built
    df["anios_remodel"] = df["yr_sold"] - df["year_remod"]
    df["area_total"] = None           # gr_liv_area + total_bsmt_sf.fillna(0)
    df["banos"] = None                # full_bath + 0.5 * half_bath
    df["calidad_cocina"] = df["kitchen_qual"].map(CALIDAD)
    df["calidad_sotano"] = df["bsmt_qual"].map(CALIDAD).fillna(0)    # sin sótano = 0
    df["garage_type"] = None          # garage_type con los NaN reemplazados por "Sin garaje"
    return df.drop(columns=["year_built", "year_remod", "full_bath", "half_bath",
                            "kitchen_qual", "bsmt_qual"])

In [ ]:
# ── VERIFICACIÓN 5.1 ─────────────────────────────────────────────────────
def media(col):
    if X_train is None: return None
    v = pd.to_numeric(preparar(X_train)[col], errors="coerce")
    return None if v.isna().all() else v.mean()
gt = None if X_train is None else preparar(X_train)["garage_type"]
r = [check("edad promedio", media("edad"), 36.8064),
     check("área total promedio", media("area_total"), 2541.3705, tol=0.01),
     check("baños promedio", media("banos"), 1.7536),
     check("casas «Sin garaje»", None if gt is None else (gt == "Sin garaje").sum(), 132, tol=0)]
print()
print("5.1 OK" if all(r) else "Revisa 5.1")

Fíjate en `kitchen_qual` y `bsmt_qual`: son categorías **con orden** (pobre < regular < típica < buena <
excelente). En vez de one-hot, las convertimos a números del 1 al 5, que conservan ese orden. Las que no
tienen orden (barrio, tipo de garaje) van con one-hot en el paso siguiente.

---
# Paso 6 · Armar el pipeline  ·  7 min

Todo lo que **se aprende de los datos** va dentro de un `Pipeline`, para que en la validación cruzada se
aprenda solo con la parte de entrenamiento de cada vuelta:

```
casa cruda ─► preparar ─► ┬─ numéricas:  rellenar con la mediana ─► escalar ─┬─► modelo ─► log(precio) ─► exp ─► USD
                          └─ categóricas: rellenar con la más común ─► one-hot ┘
```

- `ColumnTransformer` aplica un tratamiento distinto a cada grupo de columnas.
- `FunctionTransformer(preparar)` mete tu función del Paso 5 dentro del pipeline.
- `TransformedTargetRegressor` entrena con `log(precio)` y devuelve las predicciones en dólares.

### Ejercicio 6.1 — El preprocesador

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
CAT = ["neighborhood", "garage_type", "central_air", "sale_condition", "ms_zoning"]
NUM = ["gr_liv_area", "total_bsmt_sf", "lot_area", "lot_frontage", "overall_qual", "overall_cond",
       "garage_cars", "bedrooms", "fireplaces", "mas_vnr_area", "yr_sold", "edad", "anios_remodel",
       "area_total", "banos", "calidad_cocina", "calidad_sotano"]

def preprocesador():
    numerico = None      # make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
    categorico = None    # make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore"))
    if numerico is None or categorico is None:
        return None
    return ColumnTransformer([("num", numerico, NUM), ("cat", categorico, CAT)])

In [ ]:
# ── VERIFICACIÓN 6.1 ─────────────────────────────────────────────────────
pre = preprocesador()
n_cols = None if pre is None or X_train is None else pre.fit_transform(preparar(X_train)).shape[1]
r = [check("columnas que salen del preprocesador", n_cols, 66, tol=0)]
print()
print("6.1 OK" if all(r) else "Revisa 6.1")

### El ensamblador: de casa cruda a precio

Esta función junta todo. Cualquier modelo de las Clases 2 a 4 entra como `modelo`.

`handle_unknown="ignore"` importa en producción: si mañana llega una casa de un barrio que no estaba en
el entrenamiento, el one-hot la deja en ceros en vez de fallar.

In [ ]:
def armar(modelo, log=True):
    """preparar → preprocesar → modelo, entrenando con log(precio) si log=True."""
    if preprocesador() is None:
        return None
    pipe = Pipeline([("preparar", FunctionTransformer(preparar)),
                     ("preprocesar", preprocesador()),
                     ("modelo", modelo)])
    if not log:
        return pipe
    return TransformedTargetRegressor(regressor=pipe, func=np.log, inverse_func=np.exp)
print("armar() lista")

---
# Paso 7 · Comparar modelos con validación cruzada  ·  12 min

Todos los modelos del módulo, con hiperparámetros razonables (los de las Clases 3 y 4, adaptados a que
ahora el objetivo está en logaritmo). Cada uno se mide con **validación cruzada de 5 particiones** sobre
el entrenamiento: la prueba sigue guardada.

`cross_validate` entrena el pipeline completo 5 veces. En cada vuelta, la mediana para rellenar, la
escala y el one-hot se aprenden **solo** con las 4 particiones de entrenamiento de esa vuelta.

### Ejercicio 7.1 — La tabla de validación cruzada

Tarda unos 20 segundos (el Random Forest es el más lento).

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
MODELOS = {
    "Siempre la mediana": (DummyRegressor(strategy="median"), False),
    "Regresión lineal":   (LinearRegression(), True),
    "Ridge":              (Ridge(alpha=10), True),
    "Lasso":              (Lasso(alpha=0.0005, max_iter=50000), True),
    "SVR RBF":            (SVR(C=1, epsilon=0.05), True),
    "Árbol":              (DecisionTreeRegressor(min_samples_leaf=10, random_state=SEED), True),
    "Random Forest":      (RandomForestRegressor(n_estimators=200, max_features=0.5, n_jobs=-1, random_state=SEED), True),
    "XGBoost":            (XGBRegressor(n_estimators=800, learning_rate=0.03, max_depth=4, subsample=0.8,
                                        colsample_bytree=0.8, random_state=SEED), True),
}
resultados = {}
for nombre, (modelo, log) in MODELOS.items():
    r = None
    # r = cross_validate(armar(modelo, log), X_train, y_train, cv=cv,
    #                    scoring={"mae": "neg_mean_absolute_error", "mape": "neg_mean_absolute_percentage_error"})
    if r is not None:
        resultados[nombre] = {"MAE": -r["test_mae"].mean(), "± MAE": r["test_mae"].std(),
                              "MAPE": -r["test_mape"].mean()}

tabla_cv = None      # pd.DataFrame(resultados).T.sort_values("MAE")
mejor = None         # el nombre del primer modelo de tabla_cv

In [ ]:
# ── VERIFICACIÓN 7.1 ─────────────────────────────────────────────────────
r = [check_bool("evaluaste los 8 modelos", len(resultados) == 8, "descomenta el cross_validate"),
     check_bool("el mejor en validación cruzada es XGBoost", mejor == "XGBoost"),
     check("MAE de la regresión lineal", resultados.get("Regresión lineal", {}).get("MAE"), 15402.8240, tol=50),
     check("MAE de XGBoost", resultados.get("XGBoost", {}).get("MAE"), 14345.0721, tol=800)]
print()
print("7.1 OK" if all(r) else "Revisa 7.1")

**Lo que dice la tabla:**

- Cualquier modelo le gana por lejos a la vara: de 55 211 USD de error a unos 15 mil.
- **Esta vez la recta casi empata con los ensembles.** Regresión lineal, Ridge, Lasso, SVR y Random Forest
  quedan todos alrededor de 15 383; XGBoost gana por unos 1 038 USD. En California la recta perdía
  por 20 mil. ¿Qué cambió? Aquí las variables ya traen el juicio del tasador (`overall_qual`), las creamos
  con sentido (`area_total`, `edad`) y el barrio entra como categoría. **Con buenas variables, un modelo
  simple llega lejos.**
- El árbol solo es el peor de los modelos de verdad: la inestabilidad de la Clase 4.
- La columna «± MAE» es la desviación entre las 5 particiones. Diferencias más chicas que eso entre dos
  modelos son, en la práctica, un empate.

### Demostración 7.2 — ¿Sirvió el logaritmo?

In [ ]:
# ── DEMOSTRACIÓN 7.2 ─────────────────────────────────────────────────────
if armar(LinearRegression()) is not None:
    for log in [False, True]:
        r_ = cross_validate(armar(LinearRegression(), log), X_train, y_train, cv=cv,
                            scoring={"mae": "neg_mean_absolute_error", "mape": "neg_mean_absolute_percentage_error"})
        print(f"Regresión lineal {'con' if log else 'sin'} log(precio): "
              f"MAE {-r_['test_mae'].mean():,.0f} · MAPE {-100*r_['test_mape'].mean():.1f} %")

Para la regresión lineal, entrenar con el logaritmo baja el error de 18 735 a 15 403 USD. En logaritmo,
cada coeficiente se lee como un **porcentaje**: «un punto más de calidad sube el precio un tanto por ciento», que
es como funcionan los precios de las casas.

---
# Paso 8 · Evaluar una vez en prueba y dejarlo listo para usar  ·  11 min

Elegimos el modelo con validación cruzada. Ahora, y **solo ahora**:

1. Lo entrenamos con **todo** el entrenamiento.
2. Lo medimos **una vez** en la prueba, con las métricas del Paso 1.
3. Lo guardamos para usarlo con casas nuevas.

### Ejercicio 8.1 — La prueba

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
modelo_final = None     # armar(*MODELOS[mejor]) entrenado con X_train, y_train
pred_test = None        # sus predicciones en X_test
m_final = None          # metricas(y_test, pred_test)

vara = None             # armar(*MODELOS["Siempre la mediana"]) entrenado con X_train, y_train
m_vara = None           # metricas en prueba de la vara

In [ ]:
# ── VERIFICACIÓN 8.1 ─────────────────────────────────────────────────────
r = [check("MAE en prueba", None if m_final is None else m_final["MAE"], 14179.8135, tol=800),
     check("MAPE en prueba", None if m_final is None else m_final["MAPE"], 0.0803, tol=0.005),
     check("R² en prueba", None if m_final is None else m_final["R2"], 0.9237, tol=0.005),
     check("MAPE de la vara", None if m_vara is None else m_vara["MAPE"], 0.3166)]
print()
print("8.1 OK" if all(r) else "Revisa 8.1")

El MAE de prueba (14 180) queda muy cerca de lo que estimó la validación cruzada (14 345): la
estimación fue honesta. Y el error típico es de **8.0 %** del precio, frente a 31.7 % de la vara.

### Demostración 8.2 — ¿Dónde se equivoca?

Un número promedio esconde mucho. Para un comité, la pregunta es **para qué casas** es confiable.

In [ ]:
# ── DEMOSTRACIÓN 8.2 ─────────────────────────────────────────────────────
if pred_test is not None:
    error_pct = np.abs(y_test - pred_test) / y_test
    segmento = pd.qcut(y_test, 5)
    resumen = pd.DataFrame({"MAPE": error_pct.groupby(segmento).mean(),
                            "MAE (USD)": np.abs(y_test - pred_test).groupby(segmento).mean()})
    print("Error por quintil de precio real:")
    print(resumen.round({"MAPE": 3, "MAE (USD)": 0}).to_string())
    print(f"\nCasas tasadas con error ≤ 10 %: {(error_pct <= 0.10).mean():.1%}")
    print(f"Casas tasadas con error ≤ 20 %: {(error_pct <= 0.20).mean():.1%}")

    fig, ax = plt.subplots(figsize=(6, 5.5))
    ax.scatter(y_test, pred_test, s=8, alpha=0.5, color=BLUE)
    lim = [40000, 650000]
    ax.plot(lim, lim, color=MAG, lw=1.5, label="predicción perfecta")
    ax.plot(lim, [l * 1.2 for l in lim], color=GRAY, ls="--", lw=1, label="±20 %")
    ax.plot(lim, [l * 0.8 for l in lim], color=GRAY, ls="--", lw=1)
    ax.set_xlabel("precio real (USD)"); ax.set_ylabel("precio tasado por el modelo (USD)")
    ax.set_title("Prueba: real vs. tasado"); ax.legend(); plt.tight_layout(); plt.show()

- **En dólares**, el error crece con el precio: 11 203 en el quintil más barato y 26 956 en el más caro. Es esperable.
- **En porcentaje**, el peor quintil es el **más barato** (12.1 %), casi el doble que los del medio. Ahí
  hay casas en mal estado y ventas atípicas, y un error de 10 mil pesa mucho.
- 75.0 % de las casas quedan tasadas con un error de hasta 10 %, y 93.2 % con hasta 20 %.

> Eso es lo que se reporta: no «R² 0.924», sino «tres de cada cuatro casas quedan a menos de 10 %
> del precio real, y el modelo es menos confiable para las viviendas de menos de 124 mil».

### Ejercicio 8.3 — Guardar el modelo y tasar una casa nueva

El modelo guardado es **el pipeline completo**: recibe la casa con las mismas columnas crudas del Paso 2
(incluso con datos faltantes) y devuelve dólares.

In [ ]:
# ── TU CÓDIGO ────────────────────────────────────────────────────────────
nueva = pd.DataFrame([{
    "gr_liv_area": 1500, "total_bsmt_sf": 1000, "lot_area": 9000, "lot_frontage": np.nan,
    "year_built": 1995, "year_remod": 2005, "overall_qual": 7, "overall_cond": 5,
    "garage_cars": 2, "garage_type": "Attchd", "full_bath": 2, "half_bath": 1, "bedrooms": 3,
    "fireplaces": 1, "mas_vnr_area": 0, "neighborhood": "CollgCr", "kitchen_qual": "Gd",
    "central_air": "Y", "bsmt_qual": "Gd", "sale_condition": "Normal", "ms_zoning": "RL", "yr_sold": 2010}])

# joblib.dump(modelo_final, "modelo_tasacion.joblib")      # guardar
cargado = None          # joblib.load("modelo_tasacion.joblib")
tasacion = None         # el precio que predice el modelo cargado para la casa nueva (un número)

In [ ]:
# ── VERIFICACIÓN 8.3 ─────────────────────────────────────────────────────
r = [check_bool("cargaste el modelo desde el archivo", cargado is not None),
     check("tasación de la casa nueva", tasacion, 192715, tol=3000)]
print()
print("Paso 8 COMPLETO" if all(r) else "Revisa 8.3")

> **Ojo al reutilizarlo en otro notebook:** `joblib` guarda el pipeline, pero no el código de `preparar`.
> Antes de `joblib.load` hay que volver a definir la función `preparar` (y `CALIDAD`). Por eso en los
> equipos de datos esa función vive en un archivo `.py` aparte. Lo vemos en la Clase 16.

---
# Cierre

### Los 8 pasos, en una línea cada uno

| Paso | La regla |
|---|---|
| 1 · Problema | Decisión, objetivo, métrica de negocio, vara y alcance, **antes** de abrir los datos |
| 2 · Datos | Tamaño, duplicados, faltantes y la forma del objetivo. Leer el diccionario |
| 3 · Separar | La prueba se aparta antes de explorar, estratificada si el objetivo es asimétrico |
| 4 · Explorar | Solo con entrenamiento: correlaciones, casos raros, categorías |
| 5 · Limpiar y crear | Una función que no aprende de los datos; el alcance como regla explícita |
| 6 · Pipeline | Lo que se aprende (medianas, escalas, one-hot) va dentro del pipeline |
| 7 · Comparar | Varios modelos y la vara, con validación cruzada. La prueba sigue guardada |
| 8 · Evaluar y usar | Una vez en prueba, métricas de negocio, error por segmento, guardar el pipeline |

### Los números del día

| | MAE | MAPE | R² |
|---|---|---|---|
| Siempre la mediana (prueba) | 58 116 | 31.7 % | — |
| Ridge (prueba) | 15 183 | 8.8 % | 0.924 |
| **XGBoost (prueba)** | **14 180** | **8.0 %** | **0.924** |

### La tarea

Repite los 8 pasos con el dataset de **diamantes** (`Tarea_Clase05_EndToEnd.ipynb`). El enunciado, la
rúbrica y la fecha de entrega están en ese notebook.

### Próxima clase · Lunes 12/10

**Clase 6 · Métricas de clasificación y regresión logística.** Empieza el Módulo 2: dejamos de predecir
dólares y pasamos a predecir sí o no.